# Steel Surface Defect Inspector — YOLOv11 Training Notebook (Colab / Kaggle)

This notebook trains a compact Ultralytics **YOLOv11** model (`yolo11n.pt` or `yolo11s.pt`) on the **NEU-DET** steel surface defect dataset (6 classes: `crazing`, `inclusion`, `patches`, `pitted_surface`, `rolled-in_scale`, `scratches`).

### ⚠️ ÖNEMLİ: Başlamadan Önce Google Colab'da T4 GPU'yu Açın!
1. Üst menüden **Çalışma Zamanı (Runtime)** $\rightarrow$ **Çalışma zamanı türünü değiştir (Change runtime type)** seçeneğine tıklayın.
2. **Donanım hızlandırıcı (Hardware accelerator)** altından **T4 GPU** seçeneğini işaretleyip **Kaydet (Save)** butonuna basın.
3. Sağ üst köşedeki **Bağlan (Connect)** butonunun yanında **T4** yazdığını doğrulayın.

In [ ]:
# 1. Install Ultralytics + Kagglehub and verify GPU status
!pip install -q "ultralytics>=8.3.0" kagglehub pyyaml

import torch
if torch.cuda.is_available():
    print(f"✅ GPU AKTİF: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ UYARI: Şu an CPU modundasınız! Eğitim çok yavaş sürer. Lütfen üst menüden 'Çalışma Zamanı -> Çalışma zamanı türünü değiştir -> T4 GPU' seçin.")

## 2. Setup Workspace Directory

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path("/content/steel-defect-inspector") if Path("/content").exists() else Path.cwd()
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
print(f"Working directory: {PROJECT_DIR}")

## 3. Download NEU-DET Dataset

In [ ]:
import kagglehub

# Download NEU Surface Defect Database from Kaggle
raw_download_path = Path(kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database"))
print(f"Downloaded NEU-DET to: {raw_download_path}")

## 4. Convert VOC XML Annotations to YOLO Format (70 / 15 / 15 Stratified Split)

In [ ]:
import random
import shutil
import xml.etree.ElementTree as ET
from collections import defaultdict
import yaml

CLASSES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "NEU-DET"

image_map = {
    p.stem.lower(): p
    for p in raw_download_path.rglob("*")
    if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".bmp", ".png")
}
xml_map = {p.stem.lower(): p for p in raw_download_path.rglob("*.xml") if p.is_file()}
common_stems = sorted(set(image_map.keys()) & set(xml_map.keys()))
print(f"Matched {len(common_stems)} image-XML pairs.")

grouped = defaultdict(list)
for stem in common_stems:
    cls_key = next((c for c in CLASSES if stem.startswith(c) or stem.startswith(c.replace("_", "-"))), "unknown")
    grouped[cls_key].append((image_map[stem], xml_map[stem]))

rng = random.Random(42)
splits = {"train": [], "val": [], "test": []}
for cls_key in sorted(grouped.keys()):
    items = sorted(grouped[cls_key], key=lambda x: x[0].stem)
    rng.shuffle(items)
    n = len(items)
    n_train, n_val = int(round(n * 0.70)), int(round(n * 0.15))
    splits["train"].extend(items[:n_train])
    splits["val"].extend(items[n_train:n_train + n_val])
    splits["test"].extend(items[n_train + n_val:])

if PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)

for split_name, split_items in splits.items():
    img_out = PROCESSED_DIR / "images" / split_name
    lbl_out = PROCESSED_DIR / "labels" / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img_path, xml_path in split_items:
        root = ET.parse(xml_path).getroot()
        size = root.find("size")
        w = int(size.findtext("width", "200")) if size is not None else 200
        h = int(size.findtext("height", "200")) if size is not None else 200
        w, h = (w or 200), (h or 200)
        lines = []
        for obj in root.findall("object"):
            cname = obj.findtext("name", "").strip().lower().replace(" ", "_")
            if cname not in CLASS_TO_ID:
                continue
            b = obj.find("bndbox")
            xmin = max(0.0, min(float(w), float(b.findtext("xmin", "0"))))
            ymin = max(0.0, min(float(h), float(b.findtext("ymin", "0"))))
            xmax = max(0.0, min(float(w), float(b.findtext("xmax", "0"))))
            ymax = max(0.0, min(float(h), float(b.findtext("ymax", "0"))))
            if xmax > xmin and ymax > ymin:
                xc, yc = (xmin + xmax) / (2.0 * w), (ymin + ymax) / (2.0 * h)
                bw, bh = (xmax - xmin) / w, (ymax - ymin) / h
                lines.append(f"{CLASS_TO_ID[cname]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
        shutil.copy2(img_path, img_out / img_path.name)
        (lbl_out / f"{img_path.stem}.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"Split '{split_name}': {len(split_items)} images")

data_yaml_path = PROJECT_DIR / "configs" / "data.yaml"
data_yaml_path.parent.mkdir(parents=True, exist_ok=True)
data_cfg = {
    "path": str(PROCESSED_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 6,
    "names": {i: c for i, c in enumerate(CLASSES)},
}
with data_yaml_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

## 5. Train YOLOv11 Model
Automatically detects whether GPU (`0`) or CPU (`"cpu"`) is active so it never crashes with an invalid CUDA device error.

In [ ]:
import torch
from ultralytics import YOLO

BASE_MODEL = "yolo11n.pt"
RUN_NAME = "neu_det_yolo11n"
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"Training on device: {DEVICE}")

model = YOLO(BASE_MODEL)
results = model.train(
    data=str(data_yaml_path),
    epochs=80,
    imgsz=640,
    batch=16,
    device=DEVICE,
    project=str(PROJECT_DIR / "runs" / "detect"),
    name=RUN_NAME,
    exist_ok=True,
    seed=42,
    patience=20,
    plots=True,
)

train_run_dir = Path(results.save_dir)
best_weight_path = train_run_dir / "weights" / "best.pt"
print(f"✅ Best weights saved at: {best_weight_path}")

## 6. Evaluate on Held-Out Test Split (`split='test'`)
We evaluate the best checkpoint on the 15% unseen **test** set and print the overall and per-class table so you can copy the real numbers directly into `README.md`.

In [ ]:
best_model = YOLO(str(best_weight_path))

metrics = best_model.val(
    data=str(data_yaml_path),
    split="test",
    imgsz=640,
    device=DEVICE,
    project=str(PROJECT_DIR / "runs" / "detect"),
    name=f"{RUN_NAME}_test_eval",
    exist_ok=True,
    plots=True,
)

test_eval_dir = Path(metrics.save_dir)

print("\n=== Overall Test Set Metrics ===")
print(f"Precision (mp): {metrics.box.mp:.4f}")
print(f"Recall    (mr): {metrics.box.mr:.4f}")
print(f"mAP@50        : {metrics.box.map50:.4f}")
print(f"mAP@50-95     : {metrics.box.map:.4f}\n")

print("| Sınıf | Precision | Recall | mAP@50 | mAP@50-95 |")
print("| :--- | :---: | :---: | :---: | :---: |")
for i, cls_name in enumerate(CLASSES):
    p, r, ap50, ap = metrics.box.class_result(i)
    print(f"| `{cls_name}` | {p:.3f} | {r:.3f} | {ap50:.3f} | {ap:.3f} |")
print(
    f"| **Genel Ortalama (All)** | **{metrics.box.mp:.3f}** | **{metrics.box.mr:.3f}** "
    f"| **{metrics.box.map50:.3f}** | **{metrics.box.map:.3f}** |"
)

## 7. Package `best.pt` and Docs Plots for Download

In [ ]:
export_dir = PROJECT_DIR / "export_bundle"
(export_dir / "weights").mkdir(parents=True, exist_ok=True)
(export_dir / "docs").mkdir(parents=True, exist_ok=True)

shutil.copy2(best_weight_path, export_dir / "weights" / "best.pt")

for plot_name in ["confusion_matrix.png", "confusion_matrix_normalized.png", "PR_curve.png", "F1_curve.png"]:
    src_plot = test_eval_dir / plot_name
    if src_plot.exists():
        shutil.copy2(src_plot, export_dir / "docs" / plot_name)

if (train_run_dir / "results.png").exists():
    shutil.copy2(train_run_dir / "results.png", export_dir / "docs" / "results.png")

archive_path = shutil.make_archive(str(PROJECT_DIR / "training_artifacts"), "zip", root_dir=export_dir)
print(f"✅ İndirilebilir paket hazır: {archive_path}")

# Colab üzerinde otomatik indirme başlat:
try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    pass